In [ ]:
# 1. 安裝 Ultralytics (YOLO) 與 Roboflow 函式庫
!pip install ultralytics roboflow

# 2. 掛載 Google Drive
from google.colab import drive
import os
import yaml

print("請點擊彈出的視窗，允許 Google Drive 的存取權限...")
drive.mount('/content/drive')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.2/249.2 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 100.4 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.18
    Uninstalling idna-3.18:
      Successfully uninstalled idna-3.18
請點擊彈出的視窗，允許 Google Drive 的存取權限...
Mounted at /content/drive


In [ ]:
from roboflow import Roboflow
from getpass import getpass

# 執行時隱藏輸入金鑰，不將金鑰寫入 notebook
rf = Roboflow(api_key=getpass("Roboflow API key: "))
project = rf.workspace("petrecognition").project("qr-code-g5n7b")
version = project.version(1)

# 下載為 YOLOv8/v9 通用的 PyTorch 格式
dataset = version.download("yolov8")

# 自動獲取下載後的資料集絕對路徑
dataset_location = dataset.location
print(f"🎉 資料集已成功下載至：{dataset_location}")

In [ ]:
yaml_path = os.path.join(dataset_location, 'data.yaml')

with open(yaml_path, 'r') as f:
    config = yaml.safe_load(f)

# 強制重寫訓練、驗證、測試的絕對路徑
config['train'] = os.path.join(dataset_location, 'train', 'images')
config['val'] = os.path.join(dataset_location, 'valid', 'images')
if 'test' in config:
    config['test'] = os.path.join(dataset_location, 'test', 'images')

with open(yaml_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("✅ data.yaml 路徑已自動校正為絕對路徑！")

✅ data.yaml 路徑已自動校正為絕對路徑！


In [ ]:
import os
import torch
from ultralytics import YOLO

# 1. 定義雲端硬碟的永久儲存路徑
cloud_project_path = '/content/drive/MyDrive/YOLO_invoice/runs'
os.makedirs(cloud_project_path, exist_ok=True)

# 2. 💡 資工系自動防呆：偵測目前有沒有開 GPU 加速器
if torch.cuda.is_available():
    device_setting = 0
    print("🎮 偵測到內建 GPU (T4)，將以顯示卡進行高速訓練！")
else:
    device_setting = 'cpu'
    print("⚠️ 警告：目前沒有偵測到 GPU，將自動退回 CPU 模式（速度極慢）。")
    print("👉 請確認你有去上方選單：執行階段 -> 變更執行階段類型 -> 選擇 T4 GPU！")

# 3. 設定你想一次跑完的模型清單
models_to_train = {
    'yolov8n':  {'yaml': 'yolov8n.yaml',  'pt': 'yolov8n.pt'},
    'yolov9t':  {'yaml': 'yolov9t.yaml',  'pt': 'yolov9t.pt'},
    'yolov10n': {'yaml': 'yolov10n.yaml', 'pt': 'yolov10n.pt'}
}

# ====== 訓練參數統一設定 ======
epochs_setting = 50
img_size = 640
# ==============================

print("\n🚀 啟動排隊訓練流水線...")

for model_name, cfg in models_to_train.items():
    print("\n" + "="*50)
    print(f"🔥 當前進度：開始訓練 {model_name}...")
    print("="*50 + "\n")

    try:
        # 載入該版本的模型架構與預訓練權重
        model = YOLO(cfg['yaml']).load(cfg['pt'])

        # 開始訓練
        model.train(
            data=os.path.join(dataset_location, 'data.yaml'),
            epochs=epochs_setting,
            imgsz=img_size,
            device=device_setting,              # ✅ 自動切換 GPU 或 CPU
            project=cloud_project_path,
            name=f'test_{model_name}',
            conf=0.15,
            iou=0.5,
            workers=2,
            exist_ok=True
        )
        print(f"✅ {model_name} 訓練完成！成果已安全存入雲端。")

    except Exception as e:
        print(f"❌ {model_name} 訓練過程發生錯誤: {e}")
        print("將跳過此模型，繼續訓練下一個...")

print("\n🎉 恭喜！所有模型排隊流水線執行完畢。")

🎮 偵測到內建 GPU (T4)，將以顯示卡進行高速訓練！

🚀 啟動排隊訓練流水線...

🔥 當前進度：開始訓練 yolov8n...

Transferred 355/355 items from pretrained weights
Ultralytics 8.4.62 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=0.15, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/QR-Code-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.5, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.yaml, momentum=0.937, mosaic=1.0, multi_

In [ ]:
import os
import time
import torch
import pandas as pd
from ultralytics import YOLO

# ==================== 參數設定 ====================
# 1. 你的發票照片檔名，請確保它上傳在 Colab 的 /content/ 目錄下
# 如果你的照片檔名是別的（例如 image.jpg），請修改這裡：
invoice_image_path = '/content/invoice.JPG'

# 2. 雲端模型的存放路徑
cloud_project_path = '/content/drive/MyDrive/YOLO_invoice/runs'
# ==================================================

speed_results = []

# 防呆檢查：確認你有上傳發票照片
if not os.path.exists(invoice_image_path):
    print(f"❌ 找不到發票照片：{invoice_image_path}")
    print("👉 請點擊左側資料夾圖示，將發票照片上傳到 /content/ 目錄下，並確認檔名一致。")
else:
    # 定義我們要測試的三個模型名稱
    model_names = ['yolov8n', 'yolov9t', 'yolov10n']

    print("🚀 開始執行模型速度基準測試 (Speed Benchmark)...")

    for name in model_names:
        # 指向雲端硬碟裡剛剛訓練好的 best.pt
        model_path = os.path.join(cloud_project_path, f'test_{name}', 'weights', 'best.pt')

        if not os.path.exists(model_path):
            print(f"⚠️ 找不到 {name} 的雲端權重檔（可能該模型還沒訓練完或被中斷），跳過測試。")
            continue

        try:
            # 載入訓練好的模型
            model = YOLO(model_path)

            print(f"⏳ 正在測試 {name}...")

            # 1. 🔥 硬體熱身 (Warm-up)：跑 10 次不計時，讓 GPU 進入狀況
            for _ in range(10):
                _ = model(invoice_image_path, verbose=False)

            # 2. ⏱️ 正式計時：連續推論 100 次
            iterations = 100
            start_time = time.time()
            for _ in range(iterations):
                # 這裡的參數 conf=0.15, iou=0.5 必須跟你之後 App 執行時一致
                _ = model(invoice_image_path, verbose=False, conf=0.15, iou=0.5, imgsz=640)
            end_time = time.time()

            # 3. 📊 計算平均數據
            total_time_ms = (end_time - start_time) * 1000  # 轉成毫秒
            avg_time_ms = total_time_ms / iterations        # 單張照片平均毫秒
            fps = 1000 / avg_time_ms                        # 每秒可處理幾張圖

            speed_results.append({
                '模型名稱 (Model)': name,
                '平均推論時間 (毫秒/張)': round(avg_time_ms, 2),
                '預估每秒影格數 (FPS)': round(fps, 1)
            })

        except Exception as e:
            print(f"❌ 測試 {name} 時發生錯誤: {e}")

    # 4. 🚀 用 Pandas 漂亮地印出對比表格
    if speed_results:
        df_speed = pd.DataFrame(speed_results)
        # 依照速度由快到慢排序
        df_speed = df_speed.sort_values(by='平均推論時間 (毫秒/張)').reset_index(drop=True)
        print("\n" + "="*40)
        print("📊 三合一模型速度測試結果")
        print("="*40)
        display(df_speed)
    else:
        print("\n❌ 雲端路徑下沒有任何訓練好的模型可供速度測試。請等待訓練完成後再跑這格！")

🚀 開始執行模型速度基準測試 (Speed Benchmark)...
⏳ 正在測試 yolov8n...
⏳ 正在測試 yolov9t...
⏳ 正在測試 yolov10n...

📊 三合一模型速度測試結果


,模型名稱 (Model),平均推論時間 (毫秒/張),預估每秒影格數 (FPS)
0,yolov10n,21.87,45.7
1,yolov8n,30.69,32.6
2,yolov9t,43.44,23.0


In [ ]:
import time

models = {
    "YOLOv8": v8,
    "YOLOv9": v9,
    "YOLOv10": v10
}

for name, model in models.items():

    model("bus.jpg")   # warmup

    start = time.perf_counter()

    for i in range(50):
        model("bus.jpg", verbose=False)

    end = time.perf_counter()

    avg = (end-start)/50

    print(name, avg*1000, "ms")


image 1/1 /content/bus.jpg: 640x480 4 persons, 1 bus, 1 stop sign, 457.4ms
Speed: 20.6ms preprocess, 457.4ms inference, 36.7ms postprocess per image at shape (1, 3, 640, 480)
YOLOv8 216.18092262000118 ms

image 1/1 /content/bus.jpg: 640x480 4 persons, 1 bus, 264.2ms
Speed: 6.9ms preprocess, 264.2ms inference, 1.3ms postprocess per image at shape (1, 3, 640, 480)
YOLOv9 256.6086809600006 ms

image 1/1 /content/bus.jpg: 640x480 4 persons, 1 bus, 171.0ms
Speed: 3.2ms preprocess, 171.0ms inference, 0.3ms postprocess per image at shape (1, 3, 640, 480)
YOLOv10 203.58702333999872 ms
